# Dummy Agent Library 실습 노트북 (데스크탑 Ollama 버전)

코스 원문: [Dummy Agent Library](https://huggingface.co/learn/agents-course/unit1/dummy-agent-library)
원문은 HF 서버의 `moonshotai/Kimi-K2.5`를 쓰지만, 우리는 무료 크레딧이 0이라 **집 데스크탑의 Ollama(`qwen3.5:9b`)**로 같은 흐름을 따라 해요.

## 노트북 쓰는 법 (처음이면 꼭 읽기)

- 이 화면은 **칸(셀)**으로 나뉘어 있어요. 글 칸(지금 이 칸)과 코드 칸(회색 상자)이 있어요.
- 코드 칸을 클릭하고 **`Shift + Enter`** → 그 칸만 실행되고 결과가 바로 아래에 나와요. 다음 칸으로 넘어가요.
- **위에서부터 순서대로** 실행하세요. 아래 칸은 위 칸에서 만든 것(함수, 변수)을 써요.
- 칸 왼쪽의 `[*]`는 실행 중, `[3]` 같은 숫자는 실행 끝(몇 번째로 실행했는지)이라는 뜻이에요.
- 꼬였으면 메뉴 **Kernel → Restart** 후 처음 칸부터 다시 하면 돼요. (Kernel = 코드를 실제로 돌리는 파이썬 엔진)
- 코드 칸은 마음대로 고쳐 보세요. 망가지면 `git checkout -- dummy_agent_ollama.ipynb`로 원래대로 되돌릴 수 있어요.
- 각 단계 끝의 **✅ 확인할 것**을 보고, 결과가 다르면 그대로 복사해서 AI에게 붙여 주세요.

## 0단계. Ollama가 켜져 있나 확인

Ollama는 내 PC 안에서 `http://localhost:11434`라는 주소로 주문을 받는 **작은 서버**예요. (`localhost` = "이 컴퓨터 자신")
원문의 "HF 서버" 자리를 이 서버가 대신해요. 먼저 서버가 살아 있는지, 모델이 있는지 물어봐요.

In [ ]:
import json
import urllib.request

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3.5:9b"

with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("설치된 모델:", models)
print("qwen3.5:9b 있음?", MODEL in models)

✅ **확인할 것**
- `qwen3.5:9b 있음? True`가 나오면 통과.
- `ConnectionRefusedError` / `URLError`가 나오면 → Ollama가 꺼져 있는 거예요. 시작 메뉴에서 **Ollama 앱을 켜고** 다시 실행.
- `False`면 → 모델 이름이 다른 거예요. 목록에 보이는 이름으로 위 칸의 `MODEL = "..."`을 바꾸세요.

## 1단계. 모델에게 말 거는 함수 만들기 (원문의 Serverless API 자리)

원문 코드:
```python
client = InferenceClient(model="moonshotai/Kimi-K2.5", token=HF_TOKEN)
output = client.chat.completions.create(messages=..., max_tokens=..., stop=..., extra_body={'thinking': {'type': 'disabled'}})
print(output.choices[0].message.content)
```
우리는 같은 일을 하는 `chat()` 함수를 직접 만들어요. 인자 이름도 원문과 비슷하게 맞췄어요.

| 원문 | 여기 | 뜻 |
|---|---|---|
| `messages` | `messages` | system/user/assistant 메시지 목록 |
| `max_tokens` | `num_predict` | 최대 몇 토큰까지 쓸지 |
| `stop` | `stop` | 이 글자가 나오려 하면 멈춤 |
| `extra_body={'thinking': {'type': 'disabled'}}` | `"think": False` | 추론 모델의 속생각 끄기 |
| `output.choices[0].message.content` | `["message"]["content"]` | 답 본문 꺼내기 |

In [ ]:
def chat(messages, max_tokens=200, stop=None):
    options = {"num_predict": max_tokens}
    if stop:
        options["stop"] = stop
    body = {
        "model": MODEL,
        "messages": messages,
        "stream": False,   # 다 쓴 뒤 한 번에 받기
        "think": False,    # 속생각 끄기
        "options": options,
    }
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/chat",
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.loads(resp.read())["message"]["content"]

✅ **확인할 것**: 아무것도 안 나오면 정상이에요. 함수를 **정의만** 했고 아직 부르지 않았으니까요. (`def` = "이런 함수를 만들어 둔다")

이제 원문처럼 아주 간단한 질문을 해 봐요. 첫 호출은 모델을 GPU에 올리느라 몇 초~수십 초 걸릴 수 있어요.

In [ ]:
answer = chat([{"role": "user", "content": "The capital of France is"}], max_tokens=50)
print(answer)

✅ **확인할 것**: `Paris`가 들어간 답이 나오면 통과. (원문 결과: `Paris.`) 문장 모양이 달라도 괜찮아요.

## 2단계. 에이전트용 system prompt 넣기

원문 그대로의 system prompt예요. 들어 있는 것:
1. **도구 정보**: `get_weather` (위치를 받아 날씨를 알려 줌)
2. **도구 쓰는 법**: `action`(도구 이름) + `action_input`(인자)를 담은 JSON
3. **사이클 규칙**: Question → Thought → Action → Observation → … → `Final Answer:`

(`{{ }}`는 원문 그대로 둔 것. 모델에게는 중괄호 두 개로 보이지만 동작엔 큰 문제 없어요.)

In [ ]:
SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {"location": "New York"}
}}

ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
]

for m in messages:
    print(f"[{m['role']}] 글자 수 {len(m['content'])}")

✅ **확인할 것**: `[system] 글자 수 ...`, `[user] 글자 수 ...` 두 줄이 나오면 통과.
모델이 실제로 받는 건 이 두 메시지에 **채팅 템플릿(특수 토큰)**이 씌워진 글이에요. 원문 예시의 `<|start_header_id|>`는 Llama 3 것이고, qwen은 `<|im_start|>` 같은 자기 토큰을 써요. 공통점은 **비어 있는 assistant 자리로 끝난다**는 것 = "이제 네 차례".

## 3단계. 그냥 부르기 → 관찰을 지어내나?

원문에서는 모델이 Action을 쓰고 나서 **도구를 부르지도 않았는데** `Observation: ... 12°C`를 스스로 지어냈어요(환각).
우리 모델도 그러는지 봐요.

In [ ]:
print(chat(messages, max_tokens=200))

✅ **확인할 것**
- `Thought:` → `Action:` (JSON) 순서로 쓰나?
- 그 뒤에 `Observation:`을 **스스로** 써 버리나? 썼다면 그 날씨는 지어낸 거예요. 우리는 아직 날씨 도구를 만들지도 않았어요!
- 모델은 매번 조금씩 다르게 써요. 몇 번 더 실행해서(`Shift + Enter`) 비교해 보세요.
- 지어내지 않고 Action에서 멈추는 경우도 있을 수 있어요(모델마다 다름). 그래도 "멈출지 말지를 모델에게 맡기면 안 된다"는 게 요점이에요.

## 4단계. `Observation:` 직전에서 멈추기

`stop=["Observation:"]` = "이 글자가 나오려 하면 거기서 멈춰". 멈춘 자리의 `Observation:` 자체는 출력에 안 나와요.
이렇게 하면 **제어권이 모델에서 우리 코드로** 넘어와요 (Actions 섹션의 Stop and Parse 중 Stop).

In [ ]:
partial = chat(messages, max_tokens=150, stop=["Observation:"])
print(partial)

✅ **확인할 것**: Thought와 Action JSON까지만 나오고, `Observation:`이나 `Final Answer:`는 **없어야** 통과. (원문: "Much Better!")
아래 단계에서 `partial`(방금 결과를 담아 둔 변수)을 계속 써요. 이 칸을 다시 실행하면 `partial`도 새 결과로 바뀌어요.

## 5단계. 가짜 도구 만들고, 모델이 쓴 JSON 읽어서 실행하기

진짜라면 날씨 API를 부를 자리에 **고정된 문장을 돌려주는 가짜 함수**를 둬요 (원문과 같음).

In [ ]:
def get_weather(location):
    return f"the weather in {location} is sunny with low temperatures. \n"

get_weather("London")

✅ **확인할 것**: `'the weather in London is sunny with low temperatures. \n'`이 나오면 통과.
(노트북은 칸의 **마지막 줄 값**을 자동으로 보여 줘요. 그래서 `print` 없이도 나와요.)

원문은 여기서 `get_weather('London')`을 **손으로** 불렀어요. 우리는 한 걸음 더 가서, 모델이 쓴 글에서 JSON을 **꺼내 읽어요**(= Parse).

In [ ]:
start = partial.index("{")      # 첫 번째 { 위치
end = partial.rindex("}") + 1   # 마지막 } 위치 바로 뒤
blob = partial[start:end]       # 그 사이 = JSON 부분
print("꺼낸 JSON:\n", blob)

action = json.loads(blob)       # 글자 → 파이썬 사전(dict)
print("\n도구 이름:", action["action"])
print("인자:", action["action_input"])

✅ **확인할 것**: `도구 이름: get_weather`, `인자: {'location': 'London'}`이 나오면 통과.
에러(`JSONDecodeError`, `ValueError`)가 나면 → 4단계 결과의 JSON 모양이 이상한 거예요. 4단계를 다시 실행하고 여기로 오거나, 결과를 AI에게 붙여 주세요.

이제 꺼낸 이름과 인자로 **진짜(가짜) 도구를 실행**해요.

In [ ]:
TOOLS = {"get_weather": get_weather}   # 도구 이름 → 실제 함수

tool = TOOLS[action["action"]]
observation = tool(**action["action_input"])   # get_weather(location="London")과 같음
print(observation)

✅ **확인할 것**: `the weather in London is sunny with low temperatures.`가 나오면 통과. 이게 **진짜 Observation**이에요.

## 6단계. 진짜 관찰을 붙여서 다시 부르기

이 섹션의 핵심 기술이에요. **assistant 메시지 칸에** `모델이 쓴 Thought+Action` + `"Observation:\n"` + `진짜 도구 결과`를 이어 붙여 넣고 다시 불러요.
그러면 모델은 "내가 여기까지 썼구나" 하고 **그 뒤를 이어서** 써요(이어 쓰기, assistant prefill).

In [ ]:
messages_2 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
    {"role": "assistant", "content": partial + "Observation:\n" + observation},
]

print("----- 모델에게 보내는 assistant 칸 내용 -----")
print(messages_2[-1]["content"])

In [ ]:
print(chat(messages_2, max_tokens=200))

✅ **확인할 것**
- `Final Answer:` 뒤에 **sunny with low temperatures**(우리 도구가 준 답)가 나오면 통과. 3단계에서 지어낸 날씨와 비교해 보세요.
- 원문은 이어서 `Final Answer: ...`만 나왔어요. 우리 모델이 `Thought:`부터 다시 쓰거나 앞 내용을 반복해도, 최종 답이 sunny면 성공이에요.
- 이어 쓰기가 잘 안 되면(엉뚱한 답, 날씨를 다시 지어냄) 아래 **6-B**를 해 보세요. (2026-11-16 실습: Ollama + qwen3.5에선 6-A가 막히고 6-B가 됨)

### 6-B (막힐 때만). 관찰을 user 메시지로 전달하기
이어 쓰기를 지원하지 않는 모델·서버도 있어요(미확인: Ollama + qwen3.5에서 어떻게 되는지는 6단계 결과로 확인). 그럴 땐 관찰을 **새 user 메시지**로 알려 주는 방법도 있어요.

In [ ]:
messages_2b = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
    {"role": "assistant", "content": partial},
    {"role": "user", "content": "Observation:\n" + observation},
]
print(chat(messages_2b, max_tokens=200))

✅ **확인할 것**: 6단계와 6-B 중 어느 쪽이 더 깔끔하게 `Final Answer`를 냈는지 메모해 두세요.

---
## 여기까지가 원문 내용 🎉

손으로 한 일을 정리하면:

| 손으로 한 일 | 단계 | TAO에서 |
|---|---|---|
| system prompt에 도구 설명·규칙 쓰기 | 2 | 준비 |
| `stop`으로 멈추기 | 4 | Thought + Action |
| JSON 꺼내기 | 5 | Action 파싱 |
| 도구 실행 | 5 | Action 실행 |
| 결과를 Observation으로 붙여 다시 부르기 | 6 | Observation |

원문 결론처럼 **손이 많이 가요**. 질문이 복잡해서 도구를 여러 번 써야 하면 이걸 여러 바퀴 반복해야 하고요. 다음 섹션의 smolagents가 이걸 전부 대신해 줘요.

## 7단계 (도전). 이 과정을 루프로 자동화하기

원문에는 없는 보너스예요. 위에서 손으로 한 1바퀴를 **`Final Answer:`가 나올 때까지 자동으로 반복**하게 묶어요. 이게 에이전트 프레임워크가 속에서 하는 일의 뼈대예요.

- 6단계에서 확인한 대로 관찰은 **6-B 방식(새 user 메시지)**으로 보내요. (2026-11-16 실습: 6-A 이어 쓰기 방식 루프는 2바퀴부터 빈 출력 → 실패)
- 에러가 나면 `substring not found` 같은 알쏭달쏭한 말 대신, **어떻게 고치면 되는지(FORMAT_HINT)**까지 관찰로 알려 줘요.
- 도시마다 가짜 날씨를 다르게 해서, 여러 번 부르는 게 보이게 했어요.
- 0~2단계(`chat`, `SYSTEM_PROMPT`)를 먼저 실행해 둬야 해요. Kernel을 재시작했다면 위 칸들을 다시 실행하세요.

In [ ]:
# 7단계 (6-B 방식 + 친절한 에러 관찰) — 0~2단계를 먼저 실행해 둬야 함 (chat, SYSTEM_PROMPT, json)
FAKE_WEATHER = {
    "London": "sunny with low temperatures, 8°C",
    "Paris": "rainy, 14°C",
    "Seoul": "clear and cold, 2°C",
}

def get_weather(location):
    return FAKE_WEATHER.get(location, f"Error: unknown location '{location}'. Use a full city name.")

TOOLS = {"get_weather": get_weather}

# 에러가 났을 때 모델에게 "어떻게 고치면 되는지"까지 알려 주는 말
FORMAT_HINT = (
    "Write ONE action as a ```json block, for example:\n"
    '{"action": "get_weather", "action_input": {"location": "Paris"}}\n'
    "If you already know the answer, write 'Thought: I now know the final answer' and 'Final Answer: ...'."
)


def run_agent(question, max_steps=6):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        out = chat(messages, max_tokens=300, stop=["Observation:"])   # Thought + Action (또는 Final Answer)
        print(f"\n===== {step}바퀴: 모델이 쓴 글 =====\n{out.strip() or '(빈 출력)'}")
        messages.append({"role": "assistant", "content": out})        # 모델이 쓴 글을 대화에 남김

        if "Final Answer:" in out:   # 최종 답이 나오면 끝
            return out.split("Final Answer:")[-1].strip(), messages

        if "{" in out and "}" in out:   # Action JSON 꺼내서 도구 실행
            try:
                act = json.loads(out[out.index("{"): out.rindex("}") + 1])
                obs = TOOLS[act["action"]](**act["action_input"])
            except Exception as e:   # 형식이 틀려도 멈추지 말고, 에러 + 고치는 법을 관찰로 돌려줌
                obs = f"Error: could not run that action ({type(e).__name__}: {e}). " + FORMAT_HINT
        else:
            obs = "Error: no JSON action found in your message. " + FORMAT_HINT
        print(f"----- 에이전트가 보낸 관찰 (user 메시지, 6-B) -----\nObservation: {obs}")
        messages.append({"role": "user", "content": "Observation: " + obs})   # 6-B: 관찰은 새 user 메시지로

    return "(최대 횟수 도달: 답을 못 냄)", messages


final, history = run_agent("Which is warmer right now, London or Paris?")
print("\n★ 최종 답:", final)

print("\n----- 대화(messages)에 쌓인 것 -----")
for i, m in enumerate(history):
    lines = m["content"].strip().splitlines()
    print(f"{i}. [{m['role']}] {lines[0][:70] if lines else '(빈칸)'}")

✅ **확인할 것**
- 1바퀴에 London(또는 Paris), 2바퀴에 다른 도시, 3바퀴쯤 `Final Answer`가 나오면 대성공 (Paris 14°C가 더 따뜻함)
- 형식이 틀리거나 빈 출력이 나오면 → `Error: ... Write ONE action ...` 관찰을 받고 다음 바퀴에 고치는지 보세요 (Observe 섹션의 "에러도 관찰")
- 맨 아래 "대화에 쌓인 것"에서 바퀴마다 **[assistant] Thought/Action → [user] Observation**이 한 쌍씩 쌓이는 걸 확인
- 모델은 매번 조금씩 다르게 써요. 몇 번 더 실행해서 비교해 보세요

## 더 해 볼 실험 (선택)
- 마지막 줄 근처의 질문을 바꿔 보기: `run_agent("What's the weather in Seoul?")`, `run_agent("What's the weather in Tokyo?")` (← 가짜 날씨에 없는 도시: 에러를 보고 어떻게 하나?)
- 4단계에서 `max_tokens=20`으로 줄이면? (Action이 중간에 잘림 → 5단계 JSON 읽기가 실패)
- `"think": False`를 `True`로 바꾸면 답이 어떻게 달라지나? (속도도 비교)

## 다 했으면
3·6·7단계 결과를 복사해서 AI에게 붙여 주세요 → 노트 9번 "실행 결과"에 같이 정리해요.